# KokoAlert — Image Classifier Training (Notebook 05)
## MobileNetV2 on Tanzanian PCR-verified Droppings Dataset

**Zenodo record:** 4628934 (farm-labeled)

**Goal:** Train a 3-class classifier (healthy / coccidiosis / newcastle) using MobileNetV2.

**Why this matters:** The "Eye" module gives KokoAlert the ability to confirm disease from a farmer's droppings photo — critical for Coccidiosis (bloody) and Newcastle (green).

**Key challenge:** NCD (Newcastle) class has only 376 images vs 2,000+ for others. We fix this with aggressive augmentation + class weights.

**Target metric:** NCD recall ≥ 0.80 (missing a Newcastle case is worse than a false alarm).

In [1]:
import os
import shutil
import zipfile
import random
from pathlib import Path

import numpy as np
import requests
import tensorflow as tf
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Import from your src/ module
import sys
sys.path.insert(0, "../src")
from image_classifier import (
    build_image_classifier,
    compile_image_classifier,
    save_image_classifier,
    IMAGE_SIZE,
    IMAGE_CLASSES,
    IMAGE_CLASS_WEIGHTS,
)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

2026-05-09 02:26:31.284013: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-09 02:26:31.320950: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-09 02:26:31.329937: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-09 02:26:31.368878: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-09 02:26:32.939573: W tensorflow/compiler/tf2

TensorFlow version: 2.17.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


I0000 00:00:1778293596.673647   64301 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1778293596.865689   64301 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1778293596.865766   64301 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.


In [2]:
# ═══════════════════════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════════════════════

ZENODO_RECORD = "4628934"
RAW_DIR = Path("../data/raw/droppings")
PROCESSED_DIR = Path("../data/processed/droppings")
MODEL_SAVE_PATH = Path("../models/droppings_classifier.h5")

# Classes we KEEP (drop Salmonella)
KEEP_CLASSES = ["healthy", "coccidiosis", "newcastle"]
CLASS_DIRS = {
    "healthy": "healthy",
    "coccidiosis": "coccidiosis",
    "newcastle": "ncd",          # Original folder name in dataset
}

# Augmentation targets
NCD_AUGMENT_TARGET = 1500
BATCH_SIZE = 32
EPOCHS_PHASE1 = 15
EPOCHS_PHASE2 = 10

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Config loaded.")

Config loaded.


In [4]:
# ═══════════════════════════════════════════════════════════════════════════
# STEP 1: EXTRACT DOWNLOADED ZENODO ZIP FILES
# Download these manually from https://zenodo.org/record/4628934:
#   healthy.zip, cocci.zip, ncd.zip, salmo.zip
# Place them in data/raw/droppings/ before running this cell.
# ═══════════════════════════════════════════════════════════════════════════

import zipfile
from pathlib import Path

RAW_DIR = Path("../data/raw/droppings")
RAW_DIR.mkdir(parents=True, exist_ok=True)

zip_files = {
    "healthy.zip": "healthy",
    "cocci.zip": "coccidiosis",
    "ncd.zip": "newcastle",
    "salmo.zip": "salmonella",
}

for zip_name, folder_name in zip_files.items():
    zip_path = RAW_DIR / zip_name
    if zip_path.exists():
        print(f"Extracting {zip_name}...")
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(RAW_DIR)
        print(f"  Done.")
    else:
        print(f"WARNING: {zip_name} not found. Download from Zenodo 4628934.")

# Rename cocci → coccidiosis (handle case where target already exists)
cocci_folder = RAW_DIR / "cocci"
target_folder = RAW_DIR / "coccidiosis"

if cocci_folder.exists():
    if target_folder.exists():
        # Both exist — coccidiosis already has the files, just remove cocci
        import shutil
        shutil.rmtree(cocci_folder)
        print("Removed duplicate 'cocci' folder (coccidiosis already exists).")
    else:
        cocci_folder.rename(target_folder)
        print("Renamed 'cocci' → 'coccidiosis'.")

print("Extraction complete.")

Extracting healthy.zip...
  Done.
Extracting cocci.zip...
  Done.
Extracting ncd.zip...
  Done.
Removed duplicate 'cocci' folder (coccidiosis already exists).
Extraction complete.


In [5]:
# ═══════════════════════════════════════════════════════════════════════════
# STEP 2: CLEAN & ORGANISE — DROP SALMONELLA
# ═══════════════════════════════════════════════════════════════════════════

print("Cleaning dataset — keeping only:", ", ".join(KEEP_CLASSES))

if PROCESSED_DIR.exists():
    shutil.rmtree(PROCESSED_DIR)

for split in ["train", "val"]:
    for cls in KEEP_CLASSES:
        (PROCESSED_DIR / split / cls).mkdir(parents=True, exist_ok=True)

class_counts = {}

for cls_key, folder_name in CLASS_DIRS.items():
    src = RAW_DIR / folder_name
    if not src.exists():
        raise FileNotFoundError(f"Class folder not found: {src}")

    images = [f for f in src.iterdir() if f.suffix.lower() in (".jpg", ".jpeg", ".png")]
    class_counts[cls_key] = len(images)
    print(f"  {cls_key}: {len(images)} images")

    # Stratified 80/20 split
    train_imgs, val_imgs = train_test_split(
        images, test_size=0.2, random_state=SEED, shuffle=True
    )

    for img in train_imgs:
        shutil.copy2(img, PROCESSED_DIR / "train" / cls_key / img.name)
    for img in val_imgs:
        shutil.copy2(img, PROCESSED_DIR / "val" / cls_key / img.name)

print("\nCleaned and split into train/val.")
print("Class counts:", class_counts)

Cleaning dataset — keeping only: healthy, coccidiosis, newcastle
  healthy: 2057 images
  coccidiosis: 2103 images
  newcastle: 376 images

Cleaned and split into train/val.
Class counts: {'healthy': 2057, 'coccidiosis': 2103, 'newcastle': 376}


In [6]:
# ═══════════════════════════════════════════════════════════════════════════
# STEP 3: AUGMENT NCD (NEWCASTLE) CLASS
# From ~300 → ~1,500 images using rotation, flip, brightness, zoom
# ═══════════════════════════════════════════════════════════════════════════

ncd_train_dir = PROCESSED_DIR / "train" / "newcastle"
existing = list(ncd_train_dir.glob("*"))
existing_count = len(existing)
needed = NCD_AUGMENT_TARGET - existing_count

print(f"Existing NCD train images: {existing_count}")
print(f"Target: {NCD_AUGMENT_TARGET}")
print(f"To generate: {needed}")

if needed > 0:
    datagen = ImageDataGenerator(
        rotation_range=30,
        horizontal_flip=True,
        brightness_range=[0.7, 1.3],
        zoom_range=0.2,
        fill_mode="nearest",
    )

    generated = 0
    for img_path in existing:
        if generated >= needed:
            break

        img = Image.open(img_path).convert("RGB")
        img = img.resize(IMAGE_SIZE)
        img_array = np.expand_dims(np.array(img), axis=0)

        for batch in datagen.flow(img_array, batch_size=1):
            aug_img = Image.fromarray(batch[0].astype(np.uint8))
            aug_name = f"aug_{generated:04d}_{img_path.stem}.jpg"
            aug_img.save(ncd_train_dir / aug_name, quality=95)
            generated += 1
            if generated >= needed:
                break

    print(f"Generated {generated} augmented images.")
else:
    print("No augmentation needed.")

final_count = len(list(ncd_train_dir.glob("*")))
print(f"Total NCD train images now: {final_count}")

Existing NCD train images: 300
Target: 1500
To generate: 1200
Generated 1200 augmented images.
Total NCD train images now: 1500


In [7]:
# ═══════════════════════════════════════════════════════════════════════════
# STEP 4: CREATE TRAIN & VALIDATION DATA GENERATORS
# ═══════════════════════════════════════════════════════════════════════════

train_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input,
    rotation_range=15,
    horizontal_flip=True,
    zoom_range=0.1,
)

val_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input,
)

train_gen = train_datagen.flow_from_directory(
    PROCESSED_DIR / "train",
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True,
    seed=SEED,
)

val_gen = val_datagen.flow_from_directory(
    PROCESSED_DIR / "val",
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False,
)

print(f"\nTrain samples: {train_gen.samples}")
print(f"Val samples:   {val_gen.samples}")
print(f"Class indices: {train_gen.class_indices}")

Found 4827 images belonging to 3 classes.
Found 909 images belonging to 3 classes.

Train samples: 4827
Val samples:   909
Class indices: {'coccidiosis': 0, 'healthy': 1, 'newcastle': 2}


In [8]:
# ═══════════════════════════════════════════════════════════════════════════
# STEP 5: BUILD MODEL
# ═══════════════════════════════════════════════════════════════════════════

model = build_image_classifier()
model.summary()

I0000 00:00:1778294174.659511   64301 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1778294174.666571   64301 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1778294174.666683   64301 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1778294175.220669   64301 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1778294175.221440   64301 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-05-09

Model: "KokoAlert_Eye"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,339 (9.24 MB)

 Trainable params: 164,355 (642.01 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [9]:
# ═══════════════════════════════════════════════════════════════════════════
# STEP 6: PHASE 1 — TRAIN HEAD ONLY (FROZEN BACKBONE)
# ═══════════════════════════════════════════════════════════════════════════

model = compile_image_classifier(model)

callbacks_phase1 = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1
    ),
]

history1 = model.fit(
    train_gen,
    epochs=EPOCHS_PHASE1,
    validation_data=val_gen,
    class_weight=IMAGE_CLASS_WEIGHTS,
    callbacks=callbacks_phase1,
    verbose=1,
)

print("\nPhase 1 complete.")

Epoch 1/15


I0000 00:00:1778294191.671414   64723 service.cc:146] XLA service 0x7e3e60040020 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778294191.671876   64723 service.cc:154]   StreamExecutor device (0): NVIDIA GeForce GTX 1660 Ti, Compute Capability 7.5
2026-05-09 02:36:31.895477: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-05-09 02:36:33.265813: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:531] Loaded cuDNN version 8907
2026-05-09 02:36:37.395849: E external/local_xla/xla/service/slow_operation_alarm.cc:65] Trying algorithm eng3{k11=2} for conv (f32[32,32,112,112]{3,2,1,0}, u8[0]{0}) custom-call(f32[32,32,112,112]{3,2,1,0}, f32[32,1,3,3]{3,2,1,0}), window={size=3x3 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, feature_group_count=32, custom_call_target="__cudnn$convForward", backend_config={"operation_queue_id":

  2/151 ━━━━━━━━━━━━━━━━━━━━ 9s 61ms/step - accuracy: 0.3125 - loss: 2.6018  

I0000 00:00:1778294202.973888   64723 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


151/151 ━━━━━━━━━━━━━━━━━━━━ 198s 1s/step - accuracy: 0.9035 - loss: 0.4290 - val_accuracy: 0.9538 - val_loss: 0.1341 - learning_rate: 0.0010
Epoch 2/15
151/151 ━━━━━━━━━━━━━━━━━━━━ 169s 1s/step - accuracy: 0.9536 - loss: 0.1964 - val_accuracy: 0.9637 - val_loss: 0.1083 - learning_rate: 0.0010
Epoch 3/15
151/151 ━━━━━━━━━━━━━━━━━━━━ 176s 1s/step - accuracy: 0.9623 - loss: 0.1674 - val_accuracy: 0.9571 - val_loss: 0.1178 - learning_rate: 0.0010
Epoch 4/15
151/151 ━━━━━━━━━━━━━━━━━━━━ 155s 1s/step - accuracy: 0.9652 - loss: 0.1540 - val_accuracy: 0.9692 - val_loss: 0.0943 - learning_rate: 0.0010
Epoch 5/15
151/151 ━━━━━━━━━━━━━━━━━━━━ 159s 1s/step - accuracy: 0.9702 - loss: 0.1299 - val_accuracy: 0.9571 - val_loss: 0.1225 - learning_rate: 0.0010
Epoch 6/15
151/151 ━━━━━━━━━━━━━━━━━━━━ 156s 1s/step - accuracy: 0.9727 - loss: 0.1342 - val_accuracy: 0.9428 - val_loss: 0.2006 - learning_rate: 0.0010
Epoch 7/15
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 841ms/step - accuracy: 0.9806 - loss: 0.0780
Epoch

In [10]:
# DEBUG: Print all layer names
for i, layer in enumerate(model.layers):
    print(f"Layer {i}: {layer.name} | type: {type(layer).__name__}")

Layer 0: input_layer_1 | type: InputLayer
Layer 1: mobilenetv2_1.00_224 | type: Functional
Layer 2: global_average_pooling2d | type: GlobalAveragePooling2D
Layer 3: dense | type: Dense
Layer 4: dropout | type: Dropout
Layer 5: dense_1 | type: Dense


In [11]:
# ═══════════════════════════════════════════════════════════════════════════
# STEP 7: PHASE 2 — FINE-TUNE TOP 30 LAYERS
# ═══════════════════════════════════════════════════════════════════════════

# Find the MobileNetV2 sub-model robustly (works regardless of layer index)
base_model = next(l for l in model.layers if "mobilenet" in l.name.lower())
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

# Recompile with lower learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

print(f"Total MobileNetV2 layers: {len(base_model.layers)}")
print(f"Trainable layers: {sum(1 for l in base_model.layers if l.trainable)}")

callbacks_phase2 = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7, verbose=1
    ),
]

history2 = model.fit(
    train_gen,
    epochs=EPOCHS_PHASE2,
    validation_data=val_gen,
    class_weight=IMAGE_CLASS_WEIGHTS,
    callbacks=callbacks_phase2,
    verbose=1,
)

print("\nPhase 2 complete.")

Total MobileNetV2 layers: 154
Trainable layers: 30
Epoch 1/10
151/151 ━━━━━━━━━━━━━━━━━━━━ 187s 1s/step - accuracy: 0.9546 - loss: 0.3602 - val_accuracy: 0.9648 - val_loss: 0.1189 - learning_rate: 1.0000e-05
Epoch 2/10
151/151 ━━━━━━━━━━━━━━━━━━━━ 155s 1s/step - accuracy: 0.9625 - loss: 0.1657 - val_accuracy: 0.9637 - val_loss: 0.1195 - learning_rate: 1.0000e-05
Epoch 3/10
151/151 ━━━━━━━━━━━━━━━━━━━━ 159s 1s/step - accuracy: 0.9710 - loss: 0.1202 - val_accuracy: 0.9637 - val_loss: 0.1194 - learning_rate: 1.0000e-05
Epoch 5/10
151/151 ━━━━━━━━━━━━━━━━━━━━ 146s 965ms/step - accuracy: 0.9741 - loss: 0.1028 - val_accuracy: 0.9703 - val_loss: 0.0998 - learning_rate: 1.0000e-05
Epoch 6/10
151/151 ━━━━━━━━━━━━━━━━━━━━ 145s 960ms/step - accuracy: 0.9797 - loss: 0.0872 - val_accuracy: 0.9692 - val_loss: 0.0951 - learning_rate: 1.0000e-05
Epoch 7/10
151/151 ━━━━━━━━━━━━━━━━━━━━ 150s 991ms/step - accuracy: 0.9818 - loss: 0.0877 - val_accuracy: 0.9703 - val_loss: 0.0928 - learning_rate: 1.0000e-0

In [12]:
# ═══════════════════════════════════════════════════════════════════════════
# STEP 8: EVALUATE
# ═══════════════════════════════════════════════════════════════════════════

val_gen.reset()
y_pred = model.predict(val_gen, verbose=1)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = val_gen.classes

# Handle potential mismatch
min_len = min(len(y_true), len(y_pred_classes))
y_true = y_true[:min_len]
y_pred_classes = y_pred_classes[:min_len]

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(
    y_true,
    y_pred_classes,
    target_names=KEEP_CLASSES,
    digits=3,
))

# Check NCD recall specifically
report = classification_report(
    y_true, y_pred_classes, target_names=KEEP_CLASSES,
    output_dict=True, digits=3
)
ncd_recall = report["newcastle"]["recall"]

print(f"\n>>> NCD (Newcastle) Recall: {ncd_recall:.3f}")
if ncd_recall >= 0.80:
    print(">>> ✅ TARGET MET: NCD recall >= 0.80")
else:
    print(">>> ⚠️  TARGET NOT MET: NCD recall < 0.80")
    print(">>> Consider: more augmentation, longer Phase 2, or adjust class weights.")

29/29 ━━━━━━━━━━━━━━━━━━━━ 29s 871ms/step

CLASSIFICATION REPORT
              precision    recall  f1-score   support

     healthy      0.998     0.960     0.978       421
 coccidiosis      0.981     0.988     0.984       412
   newcastle      0.809     0.947     0.873        76

    accuracy                          0.971       909
   macro avg      0.929     0.965     0.945       909
weighted avg      0.974     0.971     0.972       909


>>> NCD (Newcastle) Recall: 0.947
>>> ✅ TARGET MET: NCD recall >= 0.80


In [13]:
# ═══════════════════════════════════════════════════════════════════════════
# STEP 9: SAVE MODEL
# ═══════════════════════════════════════════════════════════════════════════

print("Saving model...")
save_image_classifier(model, str(MODEL_SAVE_PATH))
print(f"✅ Model saved to: {MODEL_SAVE_PATH}")
print(f"   File size: {MODEL_SAVE_PATH.stat().st_size / 1024 / 1024:.1f} MB")

Saving model...
✅ Model saved to: ../models/droppings_classifier.h5
   File size: 22.5 MB


In [14]:
# ═══════════════════════════════════════════════════════════════════════════
# BONUS: QUICK TEST ON A SINGLE IMAGE
# ═══════════════════════════════════════════════════════════════════════════

from image_classifier import preprocess_image, predict_droppings, load_image_classifier

# Load the saved model (simulates API startup)
loaded_model = load_image_classifier(str(MODEL_SAVE_PATH))

# Pick a random validation image to test
test_class = "newcastle"
test_dir = PROCESSED_DIR / "val" / test_class
test_images = list(test_dir.glob("*"))

if test_images:
    test_img = random.choice(test_images)
    print(f"Testing on: {test_img.name} (true class: {test_class})")

    img_array = preprocess_image(str(test_img))
    result = predict_droppings(loaded_model, img_array)

    print(f"\nPredicted: {result['class']} (confidence: {result['confidence']:.3f})")
    print(f"Reliable: {result['reliable']}")
    print(f"All probs: {result['all_probabilities']}")
else:
    print("No validation images found for testing.")

Testing on: ncd.118.jpg (true class: newcastle)

Predicted: newcastle (confidence: 0.995)
Reliable: True
All probs: {'healthy': 8.155440809787251e-06, 'coccidiosis': 0.004930813331156969, 'newcastle': 0.9950609803199768}


## Training Complete ✅

Your model is saved at `models/droppings_classifier.h5`.

**What happens next:**
1. The `src/image_classifier.py` module will load this model automatically at API startup via `load_image_classifier()`.
2. The `src/pipeline.py` module will call `analyse_image()` when a farmer sends a droppings photo.
3. The `src/diagnosis_engine.py` will receive the `image_result` dict and use it to confirm or adjust the text-based droppings answer.

**If NCD recall was below 0.80:**
- Re-run the augmentation cell with a higher `NCD_AUGMENT_TARGET`
- Increase `EPOCHS_PHASE2` to 15–20
- Try unfreezing 40 layers instead of 30
- Adjust `IMAGE_CLASS_WEIGHTS` — increase the weight for class 2 (newcastle)